# SQL & PySpark – Transformation Patterns

Covers:
1. Joins (inner, left, broadcast)
2. Window functions (rank, lag, rolling average)
3. Pivot & unpivot
4. PySpark vs Spark SQL equivalence
5. Performance tips (broadcast, cache, repartition)

> Run in Azure Databricks or locally with `pip install pyspark`

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

spark = SparkSession.builder.appName('SqlPySparkDemo').getOrCreate()
spark.sparkContext.setLogLevel('WARN')

In [ ]:
# -- Sample data ----------------------------------------------------------
shipments = spark.createDataFrame([
    ('S001', '2024-01-10', 'CUST101', 'FedEx',  'EMEA',  2.5,  'on-time'),
    ('S002', '2024-01-10', 'CUST102', 'UPS',    'AMER',  3.1,  'late'),
    ('S003', '2024-01-11', 'CUST103', 'FedEx',  'AMER',  1.8,  'on-time'),
    ('S004', '2024-01-11', 'CUST104', 'DHL',    'EMEA',  4.2,  'late'),
    ('S005', '2024-01-12', 'CUST105', 'UPS',    'APAC',  2.0,  'on-time'),
    ('S006', '2024-01-12', 'CUST101', 'DHL',    'EMEA',  3.8,  'late'),
    ('S007', '2024-01-13', 'CUST103', 'FedEx',  'AMER',  1.5,  'on-time'),
    ('S008', '2024-01-13', 'CUST106', 'UPS',    'APAC',  2.9,  'on-time'),
],  ['shipment_id', 'ship_date', 'customer_id', 'carrier', 'region', 'delivery_days', 'status'])

sla = spark.createDataFrame([
    ('CUST101', 'Gold',   3.0),
    ('CUST102', 'Silver', 4.0),
    ('CUST103', 'Gold',   3.0),
    ('CUST104', 'Bronze', 5.0),
    ('CUST105', 'Silver', 4.0),
    ('CUST106', 'Gold',   3.0),
], ['customer_id', 'tier', 'max_delivery_days'])

shipments.createOrReplaceTempView('shipments')
sla.createOrReplaceTempView('sla')
shipments.show()
sla.show()

## 1. Joins

In [ ]:
# -- DataFrame API: broadcast join (sla is small → broadcast to all nodes)
joined_df = shipments.join(F.broadcast(sla), on='customer_id', how='left')
joined_df = joined_df.withColumn(
    'sla_breached',
    (F.col('delivery_days') > F.col('max_delivery_days')).cast('boolean')
)
joined_df.select('shipment_id', 'customer_id', 'tier', 'delivery_days', 'max_delivery_days', 'sla_breached').show()

# -- Equivalent Spark SQL
spark.sql("""
    SELECT
        s.shipment_id,
        s.customer_id,
        c.tier,
        s.delivery_days,
        c.max_delivery_days,
        s.delivery_days > c.max_delivery_days AS sla_breached
    FROM shipments s
    LEFT JOIN sla c USING (customer_id)
""").show()

## 2. Window Functions

In [ ]:
# Rank shipments by delivery_days within each region
w_rank = Window.partitionBy('region').orderBy('delivery_days')

# 3-row rolling average of delivery_days per carrier
w_roll = Window.partitionBy('carrier').orderBy('ship_date').rowsBetween(-2, 0)

windowed = (
    shipments
    .withColumn('rank_in_region', F.rank().over(w_rank))
    .withColumn('rolling_avg_delivery', F.round(F.avg('delivery_days').over(w_roll), 2))
    .withColumn('lag_delivery', F.lag('delivery_days', 1).over(w_roll))
)
windowed.select('shipment_id', 'carrier', 'region', 'ship_date',
                'delivery_days', 'rank_in_region', 'rolling_avg_delivery', 'lag_delivery').show()

# Equivalent SQL
spark.sql("""
    SELECT
        shipment_id, carrier, region, ship_date, delivery_days,
        RANK() OVER (PARTITION BY region ORDER BY delivery_days) AS rank_in_region,
        ROUND(AVG(delivery_days) OVER (
            PARTITION BY carrier ORDER BY ship_date
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ), 2) AS rolling_avg_delivery
    FROM shipments
""").show()

## 3. Pivot & Unpivot

In [ ]:
# Pivot: regions as rows, carriers as columns, count of shipments as values
pivot_df = (
    shipments
    .groupBy('region')
    .pivot('carrier', ['DHL', 'FedEx', 'UPS'])
    .count()
    .fillna(0)
)
print('Pivot:')
pivot_df.show()

# Unpivot (stack) back to long format
unpivot_df = pivot_df.selectExpr(
    'region',
    "stack(3, 'DHL', DHL, 'FedEx', FedEx, 'UPS', UPS) AS (carrier, shipment_count)"
).filter(F.col('shipment_count') > 0)
print('Unpivot:')
unpivot_df.show()

## 4. Performance Tips

| Tip | When to use |
|---|---|
| `F.broadcast(small_df)` | Right-side table < 10 MB; eliminates shuffle join |
| `df.cache()` | DataFrame reused multiple times in the same job |
| `df.repartition(n, col)` | Before a wide shuffle (join, groupBy) on a skewed dataset |
| `df.coalesce(n)` | Reduce partitions before writing (avoids too many small files) |
| Predicate push-down | Filter as early as possible to reduce data scanned |
| Column pruning | `select()` only required columns before wide operations |